<a href="https://colab.research.google.com/github/Nexus-Engineering-Planning/nigeria_map_tool/blob/main/pilot_notebooks_for_maps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task
Transform the data in "ag_typologies.csv" to GeoJSON format, matching the 'Area' column to LGA, state, and Senatorial fields using the geospatial data from "lga_geojson.geojson", "state_geojson.geojson", "senatorial.json", and "senatorial_constituencies_cleaned.xlsx".

## Load data

### Subtask:
Load the `ag_typologies.csv` file into a pandas DataFrame.


**Reasoning**:
Import the pandas library and load the `ag_typologies.csv` file into a DataFrame and display the first 5 rows.



In [5]:
import pandas as pd

# Load the ag_typologies.csv file into a pandas DataFrame
ag_typologies_df = pd.read_csv('/ag_typologies.csv')

## Load Geospatial Data

### Subtask:
Load the `lga_geojson.geojson`, `state_geojson.geojson`, `senatorial.json`, and `senatorial_constituencies_cleaned.xlsx` files into DataFrames.

**Reasoning**:
Load the geospatial files using `geopandas` for GeoJSON files and `pandas` for the Excel file. Display the first 5 rows of each DataFrame.

In [9]:
import geopandas as gpd
import pandas as pd

# Load the geospatial data
lga_gdf = gpd.read_file('/content/lga_geojson.geojson')
state_gdf = gpd.read_file('/content/state_geojson.geojson')

# Try reading senatorial.json by explicitly specifying the GeoJSON driver
try:
    senatorial_gdf = gpd.read_file('/content/senatorial.json', driver='GeoJSON')
except Exception as e:
    print(f"Could not read senatorial.json as GeoJSON: {e}")
    print("Attempting to read with pandas as a regular JSON file.")
    # If reading as GeoJSON fails, try reading as a regular JSON with pandas
    try:
        senatorial_gdf = pd.read_json('/content/senatorial.json')
        print("Successfully read senatorial.json as a pandas DataFrame.")
    except Exception as json_e:
        print(f"Could not read senatorial.json as JSON with pandas: {json_e}")
        senatorial_gdf = None # Set to None if both methods fail


senatorial_excel_df = pd.read_excel('/content/senatorial_constituencies_cleaned.xlsx')

# Display the first 5 rows of each GeoDataFrame/DataFrame
print("LGA GeoDataFrame:")
display(lga_gdf.head())

print("\nState GeoDataFrame:")
display(state_gdf.head())

if senatorial_gdf is not None:
    print("\nSenatorial GeoDataFrame/DataFrame:")
    display(senatorial_gdf.head())
else:
    print("\nCould not load Senatorial GeoDataFrame/DataFrame.")


print("\nSenatorial Excel DataFrame:")
display(senatorial_excel_df.head())

Could not read senatorial.json as GeoJSON: '/content/senatorial.json' not recognized as being in a supported file format.; It might help to specify the correct driver explicitly by prefixing the file path with '<DRIVER>:', e.g. 'CSV:path'.
Attempting to read with pandas as a regular JSON file.
Successfully read senatorial.json as a pandas DataFrame.
LGA GeoDataFrame:


,lganame,lgacode,statename,statecode,geometry
0,Eastern Obolo,3002,Akwa Ibom,AK,"POLYGON ((7.64122 4.48428, 7.62169 4.47325, 7...."
1,Ekeremor,6002,Bayelsa,BY,"POLYGON ((5.96322 5.0903, 5.96684 5.08214, 5.9..."
2,Degema,33008,Rivers,RI,"POLYGON ((6.95485 4.37353, 6.85818 4.39824, 6...."
3,Andoni,33005,Rivers,RI,"POLYGON ((7.41845 4.43153, 7.40667 4.43362, 7...."
4,Akpabuyo,9003,Cross River,CR,"POLYGON ((8.58399 4.8868, 8.58424 4.85488, 8.5..."



State GeoDataFrame:


,statename,statecode,geometry
0,Cross River,CR,"POLYGON ((8.46076 4.75455, 8.45885 4.75393, 8...."
1,Fct,FC,"POLYGON ((7.47762 8.63133, 7.44303 8.59328, 7...."
2,Ogun,OG,"POLYGON ((4.59755 6.36057, 4.59757 6.35688, 4...."
3,Oyo,OY,"POLYGON ((3.99896 7.12657, 3.99755 7.12617, 3...."
4,Sokoto,SO,"POLYGON ((4.63265 11.67143, 4.62886 11.6681, 4..."



Senatorial GeoDataFrame/DataFrame:


,Senatorial_District,LGAs
0,ABIA CENTRAL,IKWUANO
1,ABIA CENTRAL,ISIALA NGWA NORTH
2,ABIA CENTRAL,ISIALA NGWA SOUTH
3,ABIA CENTRAL,Isiala-Ngwa North
4,ABIA CENTRAL,Isiala-Ngwa South



Senatorial Excel DataFrame:


,Serial_Number,Senatorial_District,CODE,LGAs,State,Collation_Centre
0,1,ABIA NORTH,SD/001/AB,UMUNNEOCHI,ABIA,COUNCIL HALL OHAFIA LGA HQS
1,1,ABIA NORTH,SD/001/AB,ISUKWUATO,ABIA,COUNCIL HALL OHAFIA LGA HQS
2,1,ABIA NORTH,SD/001/AB,OHAFIA,ABIA,COUNCIL HALL OHAFIA LGA HQS
3,1,ABIA NORTH,SD/001/AB,AROCHUKWU,ABIA,COUNCIL HALL OHAFIA LGA HQS
4,1,ABIA NORTH,SD/001/AB,BENDE,ABIA,COUNCIL HALL OHAFIA LGA HQS


In [22]:
# Get the list of unique 'Area Name' values with still missing geometry after fuzzy match attempt
# This list is stored in areas_with_still_missing_geometry_after_fuzzy from cell 4752486c

# Get the list of all unique LGA names from lga_gdf
all_lga_names = lga_gdf['lganame'].unique()

# Get the list of LGA names that *were* matched by fuzzy matching
matched_lga_names_fuzzy = merged_fuzzy_lga_data[merged_fuzzy_lga_data['geometry'].notnull()]['lganame'].unique()

# Find LGA names in lga_gdf that were *not* matched by fuzzy matching
unmatched_lga_names = [lga for lga in all_lga_names if lga not in matched_lga_names_fuzzy]

# Sort both lists alphabetically
areas_with_still_missing_geometry_after_fuzzy_sorted = sorted(areas_with_still_missing_geometry_after_fuzzy)
unmatched_lga_names_sorted = sorted(unmatched_lga_names)

# Display the lists side-by-side in two columns
print("Unmatched 'Area Name' values from ag_typologies_df | Unmatched LGA names from lga_gdf")
print("-" * 80)

# Determine the maximum length of the lists for printing
max_len = max(len(areas_with_still_missing_geometry_after_fuzzy_sorted), len(unmatched_lga_names_sorted))

for i in range(max_len):
    area_name = areas_with_still_missing_geometry_after_fuzzy_sorted[i] if i < len(areas_with_still_missing_geometry_after_fuzzy_sorted) else ""
    lga_name = unmatched_lga_names_sorted[i] if i < len(unmatched_lga_names_sorted) else ""
    print(f"{area_name:<40s} | {lga_name}")

Unmatched 'Area Name' values from ag_typologies_df | Unmatched LGA names from lga_gdf
--------------------------------------------------------------------------------
ABUJAMUN                                 | AFIKPO NORTH
AFIKPO                                   | AHIAZU-MBAISE
AHIZU-MB                                 | ANIOCHA NORTH
AREWA                                    | ANIOCHA SOUTH
ASKIRA/U                                 | AREWA DANDI
BIRNIN-G                                 | ASKIRA UBA
CALABAR                                  | BIRNIN GWARI
DANKO WASAGU                             | CALABAR MUNICIPAL
EGBADONORTH                              | DAMBAM
EGBADOSOUTH                              | EMURE
EMURE/ISE/ORUN                           | ESAN NORTH-EAST
ESANNORT                                 | ESAN SOUTH-EAST
ESANSOUT                                 | ESSIEN UDIM
ESSIEN-U                                 | ETHIOPE EAST
IKANORTH                                 | IKA NORTH

In [26]:
# Filter lga_gdf for Kogi state (assuming state names are in uppercase)
kogi_lgas_df = lga_gdf[lga_gdf['statename'] == 'KOGI'].copy()

# Search for "Koton-Kafe" within the LGA names in Kogi state
# Use case-insensitive search and potentially fuzzy matching for robustness
search_name = 'KOTON-KAFE'.upper() # Standardize the search name

# Exact match search first
exact_match = kogi_lgas_df[kogi_lgas_df['lganame'] == search_name]

if not exact_match.empty:
    print(f"Exact match found for '{search_name}' in Kogi state:")
    display(exact_match)
else:
    print(f"No exact match found for '{search_name}' in Kogi state. Attempting fuzzy match.")

    # Fuzzy match search if exact match is not found
    lga_names_in_kogi = kogi_lgas_df['lganame'].unique()
    fuzzy_match = process.extractOne(search_name, lga_names_in_kogi, scorer=fuzz.ratio)

    if fuzzy_match and fuzzy_match[1] >= 80: # Using a threshold of 80 for fuzzy match
        matched_lga_name = fuzzy_match[0]
        score = fuzzy_match[1]
        print(f"Fuzzy match found: '{matched_lga_name}' with score {score:.2f} in Kogi state.")
        # Display the row for the fuzzy matched LGA
        display(kogi_lgas_df[kogi_lgas_df['lganame'] == matched_lga_name])
    else:
        print(f"No close fuzzy match found for '{search_name}' in Kogi state with threshold 80.")

No exact match found for 'KOTON-KAFE' in Kogi state. Attempting fuzzy match.
No close fuzzy match found for 'KOTON-KAFE' in Kogi state with threshold 80.


## Hybrid Matching: Manual Corrections + Fuzzy Matching

### Subtask:
Apply manual corrections and then perform fuzzy matching for remaining unmatched areas with LGA data.

**Reasoning**:
First, apply the previously defined manual corrections to the `ag_typologies_df`. Then, perform fuzzy matching on the resulting DataFrame to find matches for areas that were not manually corrected or still didn't match after the manual step. Finally, merge with `lga_gdf` using the combined results of manual and fuzzy matching.

In [27]:
from rapidfuzz import process, fuzz

# Define the manual mapping based on the user's input (combining previous and new corrections)
manual_corrections = {
    'CALABAR': 'CALABAR MUNICIPAL',
    'ABUJAMUN': 'MUNICIPAL AREA COUNCIL',
    'AFIKPO': 'AFIKPO NORTH',
    'AHIZU-MB': 'AHIAZU-MBAISE',
    'AREWA': 'AREWA DANDI',
    'BIRNIN-G': 'BIRNIN GWARI',
    'OHAFIA ABIA': 'OHAFIA',
    'SULE-TAN': 'SULE TANKARKAR',
    'TAFAWA-B': 'TAFAWA-BALEWA',
    'URUEOFFO': 'URUE OFFONG/ORUKO',
    'YALA CROSS': 'YALA',
    'YAMALTU': 'YAMALTU/DEBA',
    # New corrections from the user
    'DANKO WASAGU': 'WASAGU-DANKO', # Corrected based on the comparison list output
    'EGBADONORTH': 'YEWA NORTH',   # Corrected based on the user's input
    'EGBADOSOUTH': 'YEWA SOUTH',   # Corrected based on the user's input
    # Note: "EGBADONORTH is the same as ESAN NORTH-EAST" seems to be a conflict/error with "EgbadoNorth is same as Yewa North".
    # I will use the Yewa North mapping as it is listed first and seems more likely based on the name.
    'ESANNORT': 'ESAN NORTH-EAST', # Corrected based on the user's input
    'ESANSOUT': 'ESAN SOUTH-EAST', # Corrected based on the user's input - typo corrected from ESANNORT
    'ESSIEN-U': 'ESSIEN UDIM',    # Corrected based on the user's input
    'IKANORTH': 'IKA NORTH EAST',  # Corrected based on the user's input
    'IKOT-ABA': 'IKOT ABASI',     # Corrected based on the user's input
    'IKOT-EKP': 'IKOT EKPENE',    # Corrected based on the user's input
    'ILAJEESEODO': 'ILAJE',      # Corrected based on the user's input - assuming it maps to ILAJE
    'KANO': 'KANO MUNICIPAL',     # Corrected based on the user's input
    'KARIM-LA': 'KARIM LAMIDO',   # Corrected based on the user's input
    'KAURA-NA': 'KAURA NAMODA',   # Corrected based on the user's input
    'KOKO/BES': 'KOKO-BESSE',     # Corrected based on the user's input
    'KONTOGUR': 'KONTAGORA',      # Corrected based on the user's input
    'OGBA/EGBE': 'OGBA/EGBEMA/NDONI', # Corrected based on the user's input
    'OREDO EDO': 'OREDO',        # Corrected based on the user's input
    'KATSINA (BENUE)': 'KATSINA-ALA', # Manual correction based on comparison list
    'ASKIRA/U': 'ASKIRA UBA', # Manual correction based on comparison list
    'OBOMA NGWA': 'OBI NWGA', # Manual correction based on comparison list
    'SABON-GA': 'SABON GARI', # Manual correction based on comparison list
    'MAINLAND': 'LAGOS MAINLAND', # Manual correction based on comparison list
    'OVIANORT': 'OVIA NORTH-EAST', # Manual correction based on comparison list
    'EMURE/ISE/ORUN': 'EMURE', # Manual correction based on comparison list - closest match from list
    'LAKE CHAD': 'MARTE', # Manual correction based on previous outputs/comparison list - needs verification if MARTE is the correct LGA for "Lake Chad" region. Using MARTE for now as it appeared in a previous unmatched list with Lake Chad.
    'KATSINA (K)': 'KATSINA', # Manual correction based on comparison list
    'KOTONKAR': 'KOTON-KAFE' # Manual correction for the last unmatched LGA

}

# Apply the manual corrections to a copy of the original ag_typologies_df
ag_typologies_df_manual_corrected = ag_typologies_df.copy()
ag_typologies_df_manual_corrected['Area Name Corrected'] = ag_typologies_df_manual_corrected['Area Name'].apply(
    lambda x: manual_corrections.get(x, x)
)

# Perform an exact merge with lga_gdf using the manually corrected names
merged_after_manual_correction_exact = pd.merge(ag_typologies_df_manual_corrected, lga_gdf, left_on='Area Name Corrected', right_on='lganame', how='left')

# Identify rows that still didn't find a match after manual correction and exact merge
missing_manual_exact_match_df = merged_after_manual_correction_exact[merged_after_manual_correction_exact['geometry'].isnull()].copy()

# Get the list of 'Area Name Corrected' values from these missing rows for fuzzy matching
areas_for_fuzzy_matching = missing_manual_exact_match_df['Area Name Corrected'].unique()

# Get the list of LGA names to match against for fuzzy matching
lga_names_list = lga_gdf['lganame'].unique()

# Perform fuzzy matching for each area that was not matched manually/exactly
fuzzy_matches = []
fuzzy_match_threshold = 80 # Use the same threshold as before

for area in areas_for_fuzzy_matching:
    best_match = process.extractOne(area, lga_names_list, scorer=fuzz.ratio)
    if best_match and best_match[1] >= fuzzy_match_threshold:
        fuzzy_matches.append({'Area Name Corrected': area, 'best_lga_match_fuzzy': best_match[0], 'score': best_match[1]})

# Convert fuzzy matches to a DataFrame
fuzzy_matches_df = pd.DataFrame(fuzzy_matches)

# Create a mapping from the corrected area name to the best fuzzy match
fuzzy_match_mapping = fuzzy_matches_df.set_index('Area Name Corrected')['best_lga_match_fuzzy'].to_dict()

# Add a new column to the missing_manual_exact_match_df with the fuzzy match result
missing_manual_exact_match_df['fuzzy_lganame_match'] = missing_manual_exact_match_df['Area Name Corrected'].map(fuzzy_match_mapping)

# Now, merge the missing rows with lga_gdf using the fuzzy match key
merged_after_manual_fuzzy = pd.merge(missing_manual_exact_match_df.drop(columns=['lganame', 'lgacode', 'statename', 'statecode', 'geometry']), # Drop original geo columns
                                     lga_gdf,
                                     left_on='fuzzy_lganame_match',
                                     right_on='lganame',
                                     how='left',
                                     suffixes=('_manual_exact', '_fuzzy'))


# Combine the results: start with the manual/exact merge result and update missing geometries
# from the manual/fuzzy merge result.

# Select relevant columns from the initial manual/exact merge result
cols_from_exact_manual = merged_after_manual_correction_exact.columns.tolist()

# Select relevant columns from the manual/fuzzy merge result (those from lga_gdf)
cols_from_fuzzy_manual = ['lganame', 'lgacode', 'statename', 'statecode', 'geometry']

# Ensure column names are unique before combining
# We will use the original merged_after_manual_correction_exact as the base
# and update its missing geometry information from merged_after_manual_fuzzy

# Create a temporary dataframe from the fuzzy merge results containing only the geometry and identifier
geometry_from_fuzzy = merged_after_manual_fuzzy[['gadm36_2', 'lganame', 'lgacode', 'statename', 'statecode', 'geometry']].copy()
geometry_from_fuzzy.rename(columns={'lganame':'lganame_fuzzy', 'lgacode':'lgacode_fuzzy', 'statename':'statename_fuzzy', 'statecode':'statecode_fuzzy', 'geometry':'geometry_fuzzy'}, inplace=True)


# Merge the original manual/exact merged dataframe with the fuzzy geometry results based on gadm36_2
final_merged_hybrid = pd.merge(merged_after_manual_correction_exact, geometry_from_fuzzy, on='gadm36_2', how='left')

# Fill missing original geometry information with the fuzzy geometry information
final_merged_hybrid['geometry'] = final_merged_hybrid['geometry'].fillna(final_merged_hybrid['geometry_fuzzy'])
final_merged_hybrid['lganame'] = final_merged_hybrid['lganame'].fillna(final_merged_hybrid['lganame_fuzzy'])
final_merged_hybrid['lgacode'] = final_merged_hybrid['lgacode'].fillna(final_merged_hybrid['lgacode_fuzzy'])
final_merged_hybrid['statename'] = final_merged_hybrid['statename'].fillna(final_merged_hybrid['statename_fuzzy'])
final_merged_hybrid['statecode'] = final_merged_hybrid['statecode'].fillna(final_merged_hybrid['statecode_fuzzy'])


# Drop the temporary fuzzy columns
final_merged_hybrid = final_merged_hybrid.drop(columns=['lganame_fuzzy', 'lgacode_fuzzy', 'statename_fuzzy', 'statecode_fuzzy', 'geometry_fuzzy', 'Area Name Corrected', 'fuzzy_lganame_match'], errors='ignore')


# Display the first few rows of the final merged DataFrame after hybrid matching
print("\nFinal Merged DataFrame after hybrid (manual + fuzzy) matching with LGA data:")
display(final_merged_hybrid.head())

# Check for rows where LGA geometry is still missing after hybrid matching
missing_lga_geometry_after_hybrid = final_merged_hybrid[final_merged_hybrid['geometry'].isnull()]
print(f"\nNumber of rows with missing LGA geometry after hybrid merge: {len(missing_lga_geometry_after_hybrid)}")

# Display sample of rows with still missing LGA geometry
if not missing_lga_geometry_after_hybrid.empty:
    print("Sample rows with still missing LGA geometry after hybrid merge:")
    display(missing_lga_geometry_after_hybrid.head())

# Let's inspect the unique 'Area Name' values that still have missing geometry
areas_with_still_missing_geometry_after_hybrid = missing_lga_geometry_after_hybrid['Area Name'].unique()
print(f"\nUnique 'Area Name' values with still missing geometry after hybrid match attempt: {areas_with_still_missing_geometry_after_hybrid[:20]}") # Displaying only the first 20 unique values


Final Merged DataFrame after hybrid (manual + fuzzy) matching with LGA data:


,gadm36_2,Area Name,country_name,Country ISO3,Typology Class,Priority,Efficiency,Potential,merge_key,lganame,lgacode,statename,statecode,geometry
0,NGA.14.13_1,NSUKKA,Nigeria,NGA,Critical with moderate agricultural opportunities,52,0.051029,85453,NSUKKA,NSUKKA,14013,Enugu,EN,"POLYGON ((7.38074 6.74661, 7.37811 6.74306, 7...."
1,NGA.18.12_1,GWIWA,Nigeria,NGA,Critical with moderate agricultural opportunities,69,0.704665,73759,GWIWA,GWIWA,18009,Jigawa,JI,"POLYGON ((8.30508 12.64678, 8.2964 12.6488, 8...."
2,NGA.18.14_1,JAHUN,Nigeria,NGA,Critical with moderate agricultural opportunities,72,0.717014,91745,JAHUN,JAHUN,18026,Jigawa,JI,"POLYGON ((9.5147 11.94698, 9.50014 11.94287, 9..."
3,NGA.18.15_1,KAFINHAU,Nigeria,NGA,Critical with moderate agricultural opportunities,50,0.558918,74815,KAFINHAU,KAFIN HAUSA,18027,Jigawa,JI,"POLYGON ((10.00397 11.97663, 9.99658 11.97513,..."
4,NGA.18.16_1,KAUGAMA,Nigeria,NGA,Critical with moderate agricultural opportunities,66,0.725943,73451,KAUGAMA,KAUGAMA,18001,Jigawa,JI,"POLYGON ((9.89015 12.36739, 9.89515 12.35248, ..."



Number of rows with missing LGA geometry after hybrid merge: 1
Sample rows with still missing LGA geometry after hybrid merge:


,gadm36_2,Area Name,country_name,Country ISO3,Typology Class,Priority,Efficiency,Potential,merge_key,lganame,lgacode,statename,statecode,geometry
124,NGA.23.11_1,KOTONKAR,Nigeria,NGA,High performance,7,0.706669,141408,KOTONKAR,NaN,NaN,NaN,NaN,None



Unique 'Area Name' values with still missing geometry after hybrid match attempt: ['KOTONKAR']


## Apply Manual Corrections and Remerge with LGAs

### Subtask:
Apply the provided manual corrections to `ag_typologies_df` and re-merge with `lga_gdf`.

**Reasoning**:
Create a dictionary with the provided manual mappings. Update the 'Area Name' column in `ag_typologies_df` with these corrections for the unmatched rows. Then, perform an exact merge with `lga_gdf` using the corrected 'Area Name' values.

In [23]:
# Define the manual mapping based on the user's input
manual_corrections = {
    'CALABAR': 'CALABAR MUNICIPAL',
    'ABUJAMUN': 'MUNICIPAL AREA COUNCIL',
    'AFIKPO': 'AFIKPO NORTH',
    'AHIZU-MB': 'AHIAZU-MBAISE',
    'AREWA': 'AREWA DANDI',
    'BIRNIN-G': 'BIRNIN GWARI',
    'OHAFIA ABIA': 'OHAFIA',
    'SULE-TAN': 'SULE TANKARKAR', # Assuming Sule Tan refers to SULE TANKARKAR based on previous outputs
    'TAFAWA-B': 'TAFAWA-BALEWA',
    'URUEOFFO': 'URUE OFFONG/ORUKO',
    'YALA CROSS': 'YALA',
    'YAMALTU': 'YAMALTU/DEBA'
    # Note: 'Calabar is Calabar Municipal' is listed twice in the user input, only include it once.
    # Note: 'Arerwa is Arewa Dandi' seems to be a typo and is corrected to 'AREWA'.
    # Note: 'Sule Tan' is corrected to 'SULE-TAN' based on the unmatched list format.
}

# Create a copy of the dataframe to apply corrections
ag_typologies_df_manual_corrected = ag_typologies_df.copy()

# Apply the manual corrections to the 'Area Name' column
# We will apply this mapping to the entire column, as the mapping only contains keys
# that were in the unmatched list. Using .get() with a default of x will keep
# the original name if it's not in the manual_corrections dictionary.
ag_typologies_df_manual_corrected['Area Name'] = ag_typologies_df_manual_corrected['Area Name'].apply(
    lambda x: manual_corrections.get(x, x)
)

# Now, re-merge the manually corrected ag_typologies_df with lga_gdf using exact match
merged_after_manual_correction = pd.merge(ag_typologies_df_manual_corrected, lga_gdf, left_on='Area Name', right_on='lganame', how='left')


# Display the first few rows of the merged DataFrame after manual correction
print("\nMerged DataFrame after applying manual corrections:")
display(merged_after_manual_correction.head())

# Check for rows where LGA geometry is still missing after manual correction
missing_lga_geometry_after_manual = merged_after_manual_correction[merged_after_manual_correction['geometry'].isnull()]
print(f"\nNumber of rows with missing LGA geometry after manual correction: {len(missing_lga_geometry_after_manual)}")

# Display sample of rows with still missing LGA geometry
if not missing_lga_geometry_after_manual.empty:
    print("Sample rows with still missing LGA geometry after manual correction:")
    display(missing_lga_geometry_after_manual.head())

# Let's inspect the unique 'Area Name' values that still have missing geometry
areas_with_still_missing_geometry_after_manual = missing_lga_geometry_after_manual['Area Name'].unique()
print(f"\nUnique 'Area Name' values with still missing geometry after manual correction attempt: {areas_with_still_missing_geometry_after_manual[:20]}") # Displaying only the first 20 unique values


Merged DataFrame after applying manual corrections:


,gadm36_2,Area Name,country_name,Country ISO3,Typology Class,Priority,Efficiency,Potential,fuzzy_lganame_match,merge_key,lganame,lgacode,statename,statecode,geometry
0,NGA.14.13_1,NSUKKA,Nigeria,NGA,Critical with moderate agricultural opportunities,52,0.051029,85453,NSUKKA,NSUKKA,NSUKKA,14013,Enugu,EN,"POLYGON ((7.38074 6.74661, 7.37811 6.74306, 7...."
1,NGA.18.12_1,GWIWA,Nigeria,NGA,Critical with moderate agricultural opportunities,69,0.704665,73759,GWIWA,GWIWA,GWIWA,18009,Jigawa,JI,"POLYGON ((8.30508 12.64678, 8.2964 12.6488, 8...."
2,NGA.18.14_1,JAHUN,Nigeria,NGA,Critical with moderate agricultural opportunities,72,0.717014,91745,JAHUN,JAHUN,JAHUN,18026,Jigawa,JI,"POLYGON ((9.5147 11.94698, 9.50014 11.94287, 9..."
3,NGA.18.15_1,KAFINHAU,Nigeria,NGA,Critical with moderate agricultural opportunities,50,0.558918,74815,KAFIN HAUSA,KAFINHAU,NaN,NaN,NaN,NaN,None
4,NGA.18.16_1,KAUGAMA,Nigeria,NGA,Critical with moderate agricultural opportunities,66,0.725943,73451,KAUGAMA,KAUGAMA,KAUGAMA,18001,Jigawa,JI,"POLYGON ((9.89015 12.36739, 9.89515 12.35248, ..."



Number of rows with missing LGA geometry after manual correction: 157
Sample rows with still missing LGA geometry after manual correction:


,gadm36_2,Area Name,country_name,Country ISO3,Typology Class,Priority,Efficiency,Potential,fuzzy_lganame_match,merge_key,lganame,lgacode,statename,statecode,geometry
3,NGA.18.15_1,KAFINHAU,Nigeria,NGA,Critical with moderate agricultural opportunities,50,0.558918,74815,KAFIN HAUSA,KAFINHAU,NaN,NaN,NaN,NaN,None
8,NGA.18.21_1,MALAMMAD,Nigeria,NGA,Critical with moderate agricultural opportunities,55,0.349081,72704,MALAM MADORI,MALAMMAD,NaN,NaN,NaN,NaN,None
31,NGA.20.33_1,RIMINGAD,Nigeria,NGA,Critical with moderate agricultural opportunities,43,0.664571,97511,RIMIN GADO,RIMINGAD,NaN,NaN,NaN,NaN,None
61,NGA.21.9_1,DANMUSA,Nigeria,NGA,Critical with moderate agricultural opportunities,68,0.683697,86038,DAN MUSA,DANMUSA,NaN,NaN,NaN,NaN,None
62,NGA.3.26_1,ORUK-ANA,Nigeria,NGA,Critical with moderate agricultural opportunities,44,0.740039,88063,ORUK ANAM,ORUK-ANA,NaN,NaN,NaN,NaN,None



Unique 'Area Name' values with still missing geometry after manual correction attempt: ['KAFINHAU' 'MALAMMAD' 'RIMINGAD' 'DANMUSA' 'ORUK-ANA' 'OGU/BOLO'
 'BORSARI' 'KAURA-NA' 'DAMBAN' 'ISOKOSOU' 'ETHIOPEE' 'ISOKONOR' 'KOTONKAR'
 'ILEOLUJI/OKEIGBO' 'AKOKO SOUTH-EAST' 'AKOKO SOUTH-WEST' 'AKOKONORTHWEST'
 'ESE-ODO' 'TAMBAWAL' 'NNEWINORT']


In [33]:
# Display column names for senatorial_gdf
print("Columns in senatorial_gdf:")
if senatorial_gdf is not None:
    print(senatorial_gdf.columns.tolist())
else:
    print("senatorial_gdf could not be loaded.")

print("\nColumns in senatorial_excel_df:")
print(senatorial_excel_df.columns.tolist())

Columns in senatorial_gdf:
['Senatorial_District', 'LGAs']

Columns in senatorial_excel_df:
['Serial_Number', 'Senatorial_District', '   CODE', 'LGAs', 'State', 'Collation_Centre']


In [30]:
# Display sample of relevant columns from senatorial_gdf
print("Sample of 'LGAs' and 'Senatorial_District' from senatorial_gdf:")
if senatorial_gdf is not None and 'LGAs' in senatorial_gdf.columns and 'Senatorial_District' in senatorial_gdf.columns:
    display(senatorial_gdf[['LGAs', 'Senatorial_District']].head())
elif senatorial_gdf is not None:
    print("LGAs or Senatorial_District column not found in senatorial_gdf.")
else:
    print("senatorial_gdf could not be loaded.")


print("\nSample of 'LGAs', 'State', and 'Senatorial_District' from senatorial_excel_df:")
if 'LGAs' in senatorial_excel_df.columns and 'State' in senatorial_excel_df.columns and 'Senatorial_District' in senatorial_excel_df.columns:
    display(senatorial_excel_df[['LGAs', 'State', 'Senatorial_District']].head())
else:
     print("LGAs, State, or Senatorial_District column not found in senatorial_excel_df.")

Sample of 'LGAs' and 'Senatorial_District' from senatorial_gdf:


,LGAs,Senatorial_District
0,IKWUANO,ABIA CENTRAL
1,ISIALA NGWA NORTH,ABIA CENTRAL
2,ISIALA NGWA SOUTH,ABIA CENTRAL
3,ISIALA-NGWA NORTH,ABIA CENTRAL
4,ISIALA-NGWA SOUTH,ABIA CENTRAL



Sample of 'LGAs', 'State', and 'Senatorial_District' from senatorial_excel_df:


,LGAs,State,Senatorial_District
0,UMUNNEOCHI,ABIA,ABIA NORTH
1,ISUKWUATO,ABIA,ABIA NORTH
2,OHAFIA,ABIA,ABIA NORTH
3,AROCHUKWU,ABIA,ABIA NORTH
4,BENDE,ABIA,ABIA NORTH


In [29]:
# Group senatorial_excel_df by 'State' and count the number of unique 'Senatorial_District' in each state
senatorial_districts_per_state = senatorial_excel_df.groupby('State')['Senatorial_District'].nunique().reset_index()

# Rename the columns for clarity
senatorial_districts_per_state.columns = ['State', 'Number of Senatorial Districts']

# Sort by State name
senatorial_districts_per_state = senatorial_districts_per_state.sort_values('State')

# Display the result
print("Number of Senatorial Districts within each State:")
display(senatorial_districts_per_state)

Number of Senatorial Districts within each State:


,State,Number of Senatorial Districts
0,ABIA,3
1,ADAMAWA,3
2,AKWA IBOM,3
3,ANAMBRA,3
4,BAUCHI,3
5,BAYELSA,3
6,BENUE,3
7,BORNO,3
8,CROSS RIVER,3
9,DELTA,3


from matplotlib import pyplot as plt
senatorial_districts_per_state['Number of Senatorial Districts'].plot(kind='hist', bins=20, title='Number of Senatorial Districts')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
senatorial_districts_per_state['Number of Senatorial Districts'].plot(kind='line', figsize=(8, 4), title='Number of Senatorial Districts')
plt.gca().spines[['top', 'right']].set_visible(False)

In [32]:
# Merge with State GeoDataFrame (already done in hybrid merge, but let's ensure we have the correct columns)
# Assuming the state information (statename, statecode, geometry) is already in final_merged_hybrid from the hybrid LGA merge.
# If not, we would merge with state_gdf here. Let's check the columns of final_merged_hybrid.
print("Columns in final_merged_hybrid before merging with state/senatorial:")
print(final_merged_hybrid.columns)

# Based on the output of cell 83388d88, final_merged_hybrid already contains 'statename', 'statecode', and 'geometry' from lga_gdf merge.
# It doesn't explicitly contain state geometry from state_gdf yet.
# Let's add the state geometry by merging with state_gdf on statename.

# Since we are focusing on LGA GeoJSON, we don't necessarily need the state geometry in the primary GeoDataFrame.
# We can keep the state name and state code.
# Let's rename the LGA geometry column explicitly to avoid confusion.
final_merged_hybrid.rename(columns={'geometry': 'geometry_lga'}, inplace=True)


# Merge with Senatorial Excel DataFrame to add Senatorial District information
# We will merge based on the matched LGA name ('lganame') from the LGA merge and the 'LGAs' column in senatorial_excel_df
# We need to handle potential duplicate Senatorial Districts for a single LGA if they exist in the excel file.
# For simplicity, let's take the first occurrence if there are duplicates.

# Merge based on both LGA name and State name
merged_senatorial_by_lga_state = pd.merge(
    final_merged_hybrid,
    senatorial_excel_df[['LGAs', 'State', 'Senatorial_District']].drop_duplicates(),
    left_on=['lganame', 'statename'], # Use matched LGA name and State name from final_merged_hybrid
    right_on=['LGAs', 'State'], # Use LGAs and State from senatorial_excel_df
    how='left'
)

# The Senatorial_District column is now in merged_senatorial_by_lga_state.
# We can drop the redundant 'LGAs' and 'State' columns from the merge.
merged_senatorial_by_lga_state = merged_senatorial_by_lga_state.drop(columns=['LGAs', 'State'], errors='ignore')


# Now, let's create the GeoDataFrame
# We will use the 'geometry_lga' column as the active geometry.

# Ensure the geometry column is a GeoPandas GeometryDtype
gdf = gpd.GeoDataFrame(merged_senatorial_by_lga_state, geometry='geometry_lga')


# Set the coordinate reference system (CRS) if known. Assuming WGS84 (EPSG:4326) based on typical GeoJSON.
# You might need to confirm the correct CRS of your source data if it's different.
if gdf.crs is None:
    print("CRS not set. Assuming EPSG:4326. Please set the correct CRS if known.")
    gdf.set_crs(epsg=4326, inplace=True)
else:
    print(f"Existing CRS: {gdf.crs}")


# Display the first few rows of the GeoDataFrame
print("\nFinal GeoDataFrame:")
display(gdf.head())

# Check the number of rows in the GeoDataFrame
print(f"\nNumber of rows in the final GeoDataFrame: {len(gdf)}")

# Check for any rows with null geometry in the final GeoDataFrame (should be 0 if LGA matching was perfect)
print(f"Number of rows with null geometry in the final GeoDataFrame: {gdf['geometry_lga'].isnull().sum()}")

# Check for rows where senatorial district is still missing - Use the correct column name after merge
missing_senatorial_final = gdf[gdf['Senatorial_District_y'].isnull()]
print(f"\nNumber of rows with missing Senatorial District after merging by LGA and State: {len(missing_senatorial_final)}")

# Display some of the rows with still missing senatorial district
if not missing_senatorial_final.empty:
    print("Sample rows with still missing Senatorial District:")
    display(missing_senatorial_final.head())

# Let's inspect the unique 'Area Name' values that still have missing Senatorial District
areas_with_still_missing_senatorial = missing_senatorial_final['Area Name'].unique()
print(f"\nUnique 'Area Name' values with still missing Senatorial District: {areas_with_still_missing_senatorial[:20]}") # Displaying only the first 20 unique values

Columns in final_merged_hybrid before merging with state/senatorial:
Index(['gadm36_2', 'Area Name', 'country_name', 'Country ISO3',
       'Typology Class', 'Priority', 'Efficiency', 'Potential', 'merge_key',
       'lganame', 'lgacode', 'statename', 'statecode', 'geometry_lga',
       'Senatorial_District'],
      dtype='object')
Existing CRS: EPSG:4326

Final GeoDataFrame:


,gadm36_2,Area Name,country_name,Country ISO3,Typology Class,Priority,Efficiency,Potential,merge_key,lganame,lgacode,statename,statecode,geometry_lga,Senatorial_District_x,Senatorial_District_y
0,NGA.14.13_1,NSUKKA,Nigeria,NGA,Critical with moderate agricultural opportunities,52,0.051029,85453,NSUKKA,NSUKKA,14013,Enugu,EN,"POLYGON ((7.38074 6.74661, 7.37811 6.74306, 7....",ENUGU NORTH,NaN
1,NGA.18.12_1,GWIWA,Nigeria,NGA,Critical with moderate agricultural opportunities,69,0.704665,73759,GWIWA,GWIWA,18009,Jigawa,JI,"POLYGON ((8.30508 12.64678, 8.2964 12.6488, 8....",JIGAWA NORTH - WEST,NaN
2,NGA.18.14_1,JAHUN,Nigeria,NGA,Critical with moderate agricultural opportunities,72,0.717014,91745,JAHUN,JAHUN,18026,Jigawa,JI,"POLYGON ((9.5147 11.94698, 9.50014 11.94287, 9...",NaN,NaN
3,NGA.18.15_1,KAFINHAU,Nigeria,NGA,Critical with moderate agricultural opportunities,50,0.558918,74815,KAFINHAU,KAFIN HAUSA,18027,Jigawa,JI,"POLYGON ((10.00397 11.97663, 9.99658 11.97513,...",NaN,NaN
4,NGA.18.16_1,KAUGAMA,Nigeria,NGA,Critical with moderate agricultural opportunities,66,0.725943,73451,KAUGAMA,KAUGAMA,18001,Jigawa,JI,"POLYGON ((9.89015 12.36739, 9.89515 12.35248, ...",JIGAWA NORTH – EAST,NaN



Number of rows in the final GeoDataFrame: 785
Number of rows with null geometry in the final GeoDataFrame: 1

Number of rows with missing Senatorial District after merging by LGA and State: 785
Sample rows with still missing Senatorial District:


,gadm36_2,Area Name,country_name,Country ISO3,Typology Class,Priority,Efficiency,Potential,merge_key,lganame,lgacode,statename,statecode,geometry_lga,Senatorial_District_x,Senatorial_District_y
0,NGA.14.13_1,NSUKKA,Nigeria,NGA,Critical with moderate agricultural opportunities,52,0.051029,85453,NSUKKA,NSUKKA,14013,Enugu,EN,"POLYGON ((7.38074 6.74661, 7.37811 6.74306, 7....",ENUGU NORTH,NaN
1,NGA.18.12_1,GWIWA,Nigeria,NGA,Critical with moderate agricultural opportunities,69,0.704665,73759,GWIWA,GWIWA,18009,Jigawa,JI,"POLYGON ((8.30508 12.64678, 8.2964 12.6488, 8....",JIGAWA NORTH - WEST,NaN
2,NGA.18.14_1,JAHUN,Nigeria,NGA,Critical with moderate agricultural opportunities,72,0.717014,91745,JAHUN,JAHUN,18026,Jigawa,JI,"POLYGON ((9.5147 11.94698, 9.50014 11.94287, 9...",NaN,NaN
3,NGA.18.15_1,KAFINHAU,Nigeria,NGA,Critical with moderate agricultural opportunities,50,0.558918,74815,KAFINHAU,KAFIN HAUSA,18027,Jigawa,JI,"POLYGON ((10.00397 11.97663, 9.99658 11.97513,...",NaN,NaN
4,NGA.18.16_1,KAUGAMA,Nigeria,NGA,Critical with moderate agricultural opportunities,66,0.725943,73451,KAUGAMA,KAUGAMA,18001,Jigawa,JI,"POLYGON ((9.89015 12.36739, 9.89515 12.35248, ...",JIGAWA NORTH – EAST,NaN



Unique 'Area Name' values with still missing Senatorial District: ['NSUKKA' 'GWIWA' 'JAHUN' 'KAFINHAU' 'KAUGAMA' 'KIYAWA' 'AUYO' 'MAIGATARI'
 'MALAMMAD' 'MIGA' 'RINGIM' 'RONI' 'SULE-TAN' 'TAURA' 'YANKWASHI' 'BABURA'
 'BIRINIWA' 'BUJI' 'DUTSE' 'GAGARAWA']


## Fuzzy Matching and Merging with LGAs

### Subtask:
Perform fuzzy matching and merge `ag_typologies_df` with `lga_gdf` based on the fuzzy matches.

**Reasoning**:
Use `rapidfuzz` to find the best matching LGA name for each 'Area Name' in `ag_typologies_df`. Set a similarity threshold to consider a match valid. Then, merge the `ag_typologies_df` with the `lga_gdf` using these fuzzy matches.

In [21]:
from rapidfuzz import process, fuzz

# Get the list of LGA names to match against
lga_names_list = lga_gdf['lganame'].unique()

# Perform fuzzy matching for each 'Area Name' in ag_typologies_df
# We'll store the best match and its score
fuzzy_matches = []
for area in ag_typologies_df['Area Name']:
    # Find the best match for the area name in the list of LGA names
    best_match = process.extractOne(area, lga_names_list, scorer=fuzz.ratio)

    # best_match is a tuple: (matched_string, score, original_index)
    # We are interested in the matched_string and the score
    if best_match:
        fuzzy_matches.append({'Area Name': area, 'best_lga_match': best_match[0], 'score': best_match[1]})

# Convert the fuzzy matches to a DataFrame
fuzzy_matches_df = pd.DataFrame(fuzzy_matches)

# Set a threshold for the fuzzy match score (e.g., 80)
fuzzy_match_threshold = 80

# Filter for matches above the threshold
confident_fuzzy_matches_df = fuzzy_matches_df[fuzzy_matches_df['score'] >= fuzzy_match_threshold].copy()

print(f"\nNumber of potential fuzzy matches found with score >= {fuzzy_match_threshold}: {len(confident_fuzzy_matches_df)}")

# Display sample of confident fuzzy matches
print("\nSample of confident fuzzy matches:")
display(confident_fuzzy_matches_df.head())

# Create a mapping from the original 'Area Name' to the 'best_lga_match' for confident matches
fuzzy_match_mapping = confident_fuzzy_matches_df.set_index('Area Name')['best_lga_match'].to_dict()

# Apply this mapping to the 'Area Name' column in the original ag_typologies_df
# Create a new column to store the fuzzy matched LGA name
ag_typologies_df['fuzzy_lganame_match'] = ag_typologies_df['Area Name'].map(fuzzy_match_mapping)

# Merge the original ag_typologies_df with lga_gdf using the fuzzy matched LGA names
# We will perform a left merge to keep all rows from ag_typologies_df
# We'll merge on the 'fuzzy_lganame_match' from ag_typologies_df and 'lganame' from lga_gdf
merged_fuzzy_lga_data = pd.merge(ag_typologies_df, lga_gdf, left_on='fuzzy_lganame_match', right_on='lganame', how='left')


# Display the first few rows of the merged DataFrame after fuzzy matching
print("\nMerged DataFrame after fuzzy matching with LGA data:")
display(merged_fuzzy_lga_data.head())

# Check for rows where LGA geometry is still missing after fuzzy matching
missing_lga_geometry_after_fuzzy = merged_fuzzy_lga_data[merged_fuzzy_lga_data['geometry'].isnull()]
print(f"\nNumber of rows with missing LGA geometry after fuzzy merge: {len(missing_lga_geometry_after_fuzzy)}")

# Display sample of rows with still missing LGA geometry
if not missing_lga_geometry_after_fuzzy.empty:
    print("Sample rows with still missing LGA geometry:")
    display(missing_lga_geometry_after_fuzzy.head())

# Let's inspect the unique 'Area Name' values that still have missing geometry
areas_with_still_missing_geometry_after_fuzzy = missing_lga_geometry_after_fuzzy['Area Name'].unique()
print(f"\nUnique 'Area Name' values with still missing geometry after fuzzy match attempt: {areas_with_still_missing_geometry_after_fuzzy[:20]}") # Displaying only the first 20 unique values


Number of potential fuzzy matches found with score >= 80: 736

Sample of confident fuzzy matches:


,Area Name,best_lga_match,score
0,NSUKKA,NSUKKA,100.000000
1,GWIWA,GWIWA,100.000000
2,JAHUN,JAHUN,100.000000
3,KAFINHAU,KAFIN HAUSA,84.210526
4,KAUGAMA,KAUGAMA,100.000000



Merged DataFrame after fuzzy matching with LGA data:


,gadm36_2,Area Name,country_name,Country ISO3,Typology Class,Priority,Efficiency,Potential,fuzzy_lganame_match,merge_key,lganame,lgacode,statename,statecode,geometry
0,NGA.14.13_1,NSUKKA,Nigeria,NGA,Critical with moderate agricultural opportunities,52,0.051029,85453,NSUKKA,NSUKKA,NSUKKA,14013,Enugu,EN,"POLYGON ((7.38074 6.74661, 7.37811 6.74306, 7...."
1,NGA.18.12_1,GWIWA,Nigeria,NGA,Critical with moderate agricultural opportunities,69,0.704665,73759,GWIWA,GWIWA,GWIWA,18009,Jigawa,JI,"POLYGON ((8.30508 12.64678, 8.2964 12.6488, 8...."
2,NGA.18.14_1,JAHUN,Nigeria,NGA,Critical with moderate agricultural opportunities,72,0.717014,91745,JAHUN,JAHUN,JAHUN,18026,Jigawa,JI,"POLYGON ((9.5147 11.94698, 9.50014 11.94287, 9..."
3,NGA.18.15_1,KAFINHAU,Nigeria,NGA,Critical with moderate agricultural opportunities,50,0.558918,74815,KAFIN HAUSA,KAFINHAU,KAFIN HAUSA,18027,Jigawa,JI,"POLYGON ((10.00397 11.97663, 9.99658 11.97513,..."
4,NGA.18.16_1,KAUGAMA,Nigeria,NGA,Critical with moderate agricultural opportunities,66,0.725943,73451,KAUGAMA,KAUGAMA,KAUGAMA,18001,Jigawa,JI,"POLYGON ((9.89015 12.36739, 9.89515 12.35248, ..."



Number of rows with missing LGA geometry after fuzzy merge: 39
Sample rows with still missing LGA geometry:


,gadm36_2,Area Name,country_name,Country ISO3,Typology Class,Priority,Efficiency,Potential,fuzzy_lganame_match,merge_key,lganame,lgacode,statename,statecode,geometry
12,NGA.18.25_1,SULE-TAN,Nigeria,NGA,Critical with moderate agricultural opportunities,57,0.781207,61098,NaN,SULE-TAN,NaN,NaN,NaN,NaN,None
82,NGA.37.8_1,KAURA-NA,Nigeria,NGA,Critical with moderate agricultural opportunities,55,0.677568,89043,NaN,KAURA-NA,NaN,NaN,NaN,NaN,None
114,NGA.16.11_1,YAMALTU,Nigeria,NGA,High performance,23,0.748833,119383,NaN,YAMALTU,NaN,NaN,NaN,NaN,None
124,NGA.23.11_1,KOTONKAR,Nigeria,NGA,High performance,7,0.706669,141408,NaN,KOTONKAR,NaN,NaN,NaN,NaN,None
207,NGA.11.2_1,AFIKPO,Nigeria,NGA,High priority,75,0.126520,204630,NaN,AFIKPO,NaN,NaN,NaN,NaN,None



Unique 'Area Name' values with still missing geometry after fuzzy match attempt: ['SULE-TAN' 'KAURA-NA' 'YAMALTU' 'KOTONKAR' 'AFIKPO' 'BIRNIN-G'
 'DANKO WASAGU' 'KONTOGUR' 'KARIM-LA' 'TAFAWA-B' 'KATSINA (BENUE)'
 'ASKIRA/U' 'YALA CROSS' 'OBOMA NGWA' 'AHIZU-MB' 'SABON-GA' 'KANO'
 'MAINLAND' 'ILAJEESEODO' 'CALABAR']


## Fuzzy Matching for LGAs

### Subtask:
Implement fuzzy matching between 'Area Name' in `ag_typologies_df` and 'lganame' in `lga_gdf`.

**Reasoning**:
Install the `rapidfuzz` library for efficient fuzzy string matching.

In [20]:
# Install fuzzy matching library
%pip install rapidfuzz

## Fuzzy Matching for LGAs

### Subtask:
Implement fuzzy matching between 'Area Name' in `ag_typologies_df` and 'lganame' in `lga_gdf` for unmatched areas.

In [38]:
# Identify rows with missing Senatorial District from the final GeoDataFrame
missing_senatorial_final_df = gdf_with_fuzzy_senatorial[gdf_with_fuzzy_senatorial['Senatorial_District_y'].isnull()].copy()

# Get the unique LGA names from these missing rows, filtering out non-string values (like NaN)
lga_names_missing_senatorial = missing_senatorial_final_df['lganame'].dropna().unique()

# Get the unique LGA names from the senatorial_excel_df, filtering out non-string values (like NaN)
senatorial_excel_lga_names = senatorial_excel_df['LGAs'].dropna().unique()

# Get the unique 'Area Name' values corresponding to the missing Senatorial Districts
areas_with_still_missing_senatorial_fuzzy = missing_senatorial_final_df['Area Name'].unique()


# Sort the LGA names lists alphabetically for easier comparison
lga_names_missing_senatorial_sorted = sorted(lga_names_missing_senatorial)
senatorial_excel_lga_names_sorted = sorted(senatorial_excel_lga_names)

# Display the two lists of LGA names side-by-side in two columns
print("LGA Names (Missing Senatorial District) | LGA Names (Senatorial Excel)")
print("-" * 60)

# Determine the maximum length of the lists for printing
max_len = max(len(lga_names_missing_senatorial_sorted), len(senatorial_excel_lga_names_sorted))

for i in range(max_len):
    lga_missing = lga_names_missing_senatorial_sorted[i] if i < len(lga_names_missing_senatorial_sorted) else ""
    lga_senatorial = senatorial_excel_lga_names_sorted[i] if i < len(senatorial_excel_lga_names_sorted) else ""
    print(f"{lga_missing:<30s} | {lga_senatorial}")

print("\nUnique 'Area Name' values corresponding to the 21 rows with missing Senatorial Districts:")
for area in areas_with_still_missing_senatorial_fuzzy:
    print(area)

LGA Names (Missing Senatorial District) | LGA Names (Senatorial Excel)
------------------------------------------------------------
AREWA DANDI                    | ABA NORTH
DAWAKIN KUDU                   | ABA SOUTH
DAWAKIN TOFA                   | ABADAM
EDATI                          | ABAJI AREA COUNCIL
GIREI                          | ABAK
GWAGWALADA                     | ABAKALIKI
KABBA/BUNU                     | ABEOKUTA NORTH
KAFIN HAUSA                    | ABEOKUTA SOUTH
KAURA NAMODA                   | ABI
KIRI KASAMA                    | ABOH MBAISE
KUJE                           | ABUA-ODUAL
KWAYA KUSAR                    | ADAVI
MAGAMA                         | ADO
MAIDUGURI                      | ADO EKITI
MALAM MADORI                   | ADO-ODO/OTA
MASHEGU                        | AFIJIO
OSISIOMA NGWA                  | AFIKPO NORTH
SULE TANKARKAR                 | AFIKPO SOUTH
TALATA MAFARA                  | AGAIE
ZANGON KATAF                   | AGATU
             

In [34]:
# Display sample of relevant columns from senatorial_gdf
print("Sample of 'LGAs' and 'Senatorial_District' from senatorial_gdf:")
if senatorial_gdf is not None and 'LGAs' in senatorial_gdf.columns and 'Senatorial_District' in senatorial_gdf.columns:
    display(senatorial_gdf[['LGAs', 'Senatorial_District']].head())
elif senatorial_gdf is not None:
    print("LGAs or Senatorial_District column not found in senatorial_gdf.")
else:
    print("senatorial_gdf could not be loaded.")


print("\nSample of 'LGAs', 'State', and 'Senatorial_District' from senatorial_excel_df:")
if 'LGAs' in senatorial_excel_df.columns and 'State' in senatorial_excel_df.columns and 'Senatorial_District' in senatorial_excel_df.columns:
    display(senatorial_excel_df[['LGAs', 'State', 'Senatorial_District']].head())
else:
     print("LGAs, State, or Senatorial_District column not found in senatorial_excel_df.")

Sample of 'LGAs' and 'Senatorial_District' from senatorial_gdf:


,LGAs,Senatorial_District
0,IKWUANO,ABIA CENTRAL
1,ISIALA NGWA NORTH,ABIA CENTRAL
2,ISIALA NGWA SOUTH,ABIA CENTRAL
3,ISIALA-NGWA NORTH,ABIA CENTRAL
4,ISIALA-NGWA SOUTH,ABIA CENTRAL



Sample of 'LGAs', 'State', and 'Senatorial_District' from senatorial_excel_df:


,LGAs,State,Senatorial_District
0,UMUNNEOCHI,ABIA,ABIA NORTH
1,ISUKWUATO,ABIA,ABIA NORTH
2,OHAFIA,ABIA,ABIA NORTH
3,AROCHUKWU,ABIA,ABIA NORTH
4,BENDE,ABIA,ABIA NORTH


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


## Fuzzy Matching for Missing Senatorial Districts

### Subtask:
Apply fuzzy matching to assign Senatorial Districts to rows where it is currently missing.

**Reasoning**:
Identify rows with missing Senatorial Districts. For these rows, perform fuzzy matching on their LGA names against the 'LGAs' in `senatorial_excel_df` to find potential matches and assign the corresponding Senatorial District.

In [35]:
from rapidfuzz import process, fuzz

# Identify rows with missing Senatorial District (using the result from cell 6e9076c7)
missing_senatorial_df = gdf[gdf['Senatorial_District_y'].isnull()].copy()

# Get the list of LGA names in senatorial_excel_df for fuzzy matching
senatorial_lga_names = senatorial_excel_df['LGAs'].unique()

# Perform fuzzy matching for the LGA name of each row with a missing Senatorial District
fuzzy_senatorial_matches = []
fuzzy_match_threshold_senatorial = 80 # Threshold for senatorial fuzzy matching

for index, row in missing_senatorial_df.iterrows():
    lga_name = row['lganame'] # Use the matched LGA name from the hybrid merge
    if pd.notnull(lga_name): # Only perform fuzzy match if LGA name is not null
        best_match = process.extractOne(lga_name, senatorial_lga_names, scorer=fuzz.ratio)
        if best_match and best_match[1] >= fuzzy_match_threshold_senatorial:
            # If a confident fuzzy match is found, find the corresponding Senatorial District in senatorial_excel_df
            matched_lga_in_senatorial_df = senatorial_excel_df[senatorial_excel_df['LGAs'] == best_match[0]]
            if not matched_lga_in_senatorial_df.empty:
                senatorial_district = matched_lga_in_senatorial_df['Senatorial_District'].iloc[0] # Get the first Senatorial District if multiple
                fuzzy_senatorial_matches.append({
                    'gadm36_2': row['gadm36_2'], # Use gadm36_2 as identifier
                    'matched_senatorial_district': senatorial_district,
                    'fuzzy_lga_match_score': best_match[1]
                })

# Convert fuzzy senatorial matches to a DataFrame
fuzzy_senatorial_matches_df = pd.DataFrame(fuzzy_senatorial_matches)

# Display sample of fuzzy senatorial matches
print("\nSample of fuzzy senatorial matching results:")
display(fuzzy_senatorial_matches_df.head())

# Merge the fuzzy senatorial matches back to the main GeoDataFrame (gdf)
# We will update the 'Senatorial_District_y' column where it is currently null
# Use gadm36_2 as the key for merging

# Create a temporary column in gdf with gadm36_2 for merging
gdf['gadm36_2_merge'] = gdf['gadm36_2']

# Merge gdf with the fuzzy senatorial matches
gdf_with_fuzzy_senatorial = pd.merge(
    gdf,
    fuzzy_senatorial_matches_df,
    left_on='gadm36_2_merge',
    right_on='gadm36_2',
    how='left',
    suffixes=('_original', '_fuzzy_senatorial')
)

# Fill the missing Senatorial_District_y with the matched_senatorial_district from fuzzy matching
gdf_with_fuzzy_senatorial['Senatorial_District_y'] = gdf_with_fuzzy_senatorial['Senatorial_District_y'].fillna(gdf_with_fuzzy_senatorial['matched_senatorial_district'])

# Drop the temporary merge key and fuzzy senatorial columns
gdf_with_fuzzy_senatorial = gdf_with_fuzzy_senatorial.drop(columns=['gadm36_2_merge', 'gadm36_2_fuzzy_senatorial', 'matched_senatorial_district', 'fuzzy_lga_match_score'], errors='ignore')


# Display the first few rows of the GeoDataFrame after fuzzy senatorial matching
print("\nGeoDataFrame after fuzzy matching for Senatorial Districts:")
display(gdf_with_fuzzy_senatorial.head())

# Check for rows where senatorial district is still missing
missing_senatorial_final_fuzzy = gdf_with_fuzzy_senatorial[gdf_with_fuzzy_senatorial['Senatorial_District_y'].isnull()]
print(f"\nNumber of rows with missing Senatorial District after fuzzy senatorial merge: {len(missing_senatorial_final_fuzzy)}")

# Display some of the rows with still missing senatorial district
if not missing_senatorial_final_fuzzy.empty:
    print("Sample rows with still missing Senatorial District after fuzzy senatorial merge:")
    display(missing_senatorial_final_fuzzy.head())

# Let's inspect the unique 'Area Name' values that still have missing Senatorial District
areas_with_still_missing_senatorial_fuzzy = missing_senatorial_final_fuzzy['Area Name'].unique()
print(f"\nUnique 'Area Name' values with still missing Senatorial District after fuzzy senatorial match attempt: {areas_with_still_missing_senatorial_fuzzy[:20]}") # Displaying only the first 20 unique values


Sample of fuzzy senatorial matching results:


,gadm36_2,matched_senatorial_district,fuzzy_lga_match_score
0,NGA.14.13_1,ENUGU NORTH,100.000000
1,NGA.18.12_1,JIGAWA NORTH - WEST,100.000000
2,NGA.18.14_1,JIGAWA SOUTH – WEST,88.888889
3,NGA.18.16_1,JIGAWA NORTH – EAST,100.000000
4,NGA.18.19_1,JIGAWA SOUTH – WEST,100.000000



GeoDataFrame after fuzzy matching for Senatorial Districts:


,gadm36_2_original,Area Name,country_name,Country ISO3,Typology Class,Priority,Efficiency,Potential,merge_key,lganame,lgacode,statename,statecode,geometry_lga,Senatorial_District_x,Senatorial_District_y
0,NGA.14.13_1,NSUKKA,Nigeria,NGA,Critical with moderate agricultural opportunities,52,0.051029,85453,NSUKKA,NSUKKA,14013,Enugu,EN,"POLYGON ((7.38074 6.74661, 7.37811 6.74306, 7....",ENUGU NORTH,ENUGU NORTH
1,NGA.18.12_1,GWIWA,Nigeria,NGA,Critical with moderate agricultural opportunities,69,0.704665,73759,GWIWA,GWIWA,18009,Jigawa,JI,"POLYGON ((8.30508 12.64678, 8.2964 12.6488, 8....",JIGAWA NORTH - WEST,JIGAWA NORTH - WEST
2,NGA.18.14_1,JAHUN,Nigeria,NGA,Critical with moderate agricultural opportunities,72,0.717014,91745,JAHUN,JAHUN,18026,Jigawa,JI,"POLYGON ((9.5147 11.94698, 9.50014 11.94287, 9...",NaN,JIGAWA SOUTH – WEST
3,NGA.18.15_1,KAFINHAU,Nigeria,NGA,Critical with moderate agricultural opportunities,50,0.558918,74815,KAFINHAU,KAFIN HAUSA,18027,Jigawa,JI,"POLYGON ((10.00397 11.97663, 9.99658 11.97513,...",NaN,NaN
4,NGA.18.16_1,KAUGAMA,Nigeria,NGA,Critical with moderate agricultural opportunities,66,0.725943,73451,KAUGAMA,KAUGAMA,18001,Jigawa,JI,"POLYGON ((9.89015 12.36739, 9.89515 12.35248, ...",JIGAWA NORTH – EAST,JIGAWA NORTH – EAST



Number of rows with missing Senatorial District after fuzzy senatorial merge: 21
Sample rows with still missing Senatorial District after fuzzy senatorial merge:


,gadm36_2_original,Area Name,country_name,Country ISO3,Typology Class,Priority,Efficiency,Potential,merge_key,lganame,lgacode,statename,statecode,geometry_lga,Senatorial_District_x,Senatorial_District_y
3,NGA.18.15_1,KAFINHAU,Nigeria,NGA,Critical with moderate agricultural opportunities,50,0.558918,74815,KAFINHAU,KAFIN HAUSA,18027,Jigawa,JI,"POLYGON ((10.00397 11.97663, 9.99658 11.97513,...",NaN,NaN
8,NGA.18.21_1,MALAMMAD,Nigeria,NGA,Critical with moderate agricultural opportunities,55,0.349081,72704,MALAMMAD,MALAM MADORI,18017,Jigawa,JI,"POLYGON ((9.96439 12.41346, 9.96408 12.41283, ...",NaN,NaN
12,NGA.18.25_1,SULE-TAN,Nigeria,NGA,Critical with moderate agricultural opportunities,57,0.781207,61098,SULE-TAN,SULE TANKARKAR,18030,Jigawa,JI,"POLYGON ((8.99995 12.58723, 9.00444 12.59518, ...",NaN,NaN
82,NGA.37.8_1,KAURA-NA,Nigeria,NGA,Critical with moderate agricultural opportunities,55,0.677568,89043,KAURA-NA,KAURA NAMODA,808,Zamfara,ZA,"POLYGON ((6.77205 12.38686, 6.77064 12.38144, ...",NaN,NaN
111,NGA.15.5_1,KUJE,Nigeria,NGA,High performance,25,0.715457,171988,KUJE,KUJE,15004,Fct,FC,"POLYGON ((6.98081 8.44373, 6.98254 8.45176, 6....",NaN,NaN



Unique 'Area Name' values with still missing Senatorial District after fuzzy senatorial match attempt: ['KAFINHAU' 'MALAMMAD' 'SULE-TAN' 'KAURA-NA' 'KUJE' 'KOTONKAR' 'MASHEGU'
 'EDATI' 'KIRIKASA' 'GIRIE' 'MAGAMA' 'TALATA-MAFARA' 'OSISIOMA NGWA'
 'MAIDUGUR' 'KABBA/BU' 'GWAGWALA' 'ZANGONKA' 'KWAYA KUSAR' 'DAWAKINT'
 'DAWAKINK']
